# Preparación de Datos - 3 Tablas Estructuradas

Este notebook procesa `job_listings.json` y crea 3 tablas conectadas por `job_id`:

1. **job_descriptions.csv**: Información textual de cada job (title, summary, tasks)
2. **job_skills.csv**: Skills extraídas por job (hard y soft skills)
3. **job_balancer.csv**: Información de balanceo para ML (num_candidates, weight, frequency_group)

## Transformaciones Aplicadas:
- ✅ Lowercase en todos los textos
- ✅ Incluye hard skills (type_skill=0) y soft skills (type_skill=1)
- ✅ Sin limpieza de caracteres especiales
- ✅ Sin remoción de junior/senior
- ✅ TASKS parseados correctamente (sin formato de diccionario)
- ✅ Columna 'description' eliminada (mayoría vacía)

## 1. Carga de Datos

In [1]:
import json
import pandas as pd
import numpy as np
import ast
import os
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Librerías importadas")

✅ Librerías importadas


In [2]:
# Cargar job_listings.json
print("Cargando job_listings.json...")
with open('../Data/job_listings.json', 'r', encoding='utf-8') as f:
    job_listings = json.load(f)

print(f"✅ {len(job_listings):,} jobs cargados")

# Cargar datos de entrenamiento para balanceo
print("\nCargando datos de entrenamiento...")
y_train = pd.read_csv('../Data/y_train_SwJNMSu.csv')

print(f"✅ {len(y_train):,} registros en y_train")
print(f"   Jobs únicos en y_train: {y_train['job_id'].nunique():,}")

Cargando job_listings.json...
✅ 21,917 jobs cargados

Cargando datos de entrenamiento...
✅ 15,882 registros en y_train
   Jobs únicos en y_train: 7,030


## 2. Tabla 1: job_descriptions

Extrae información textual de cada job:
- job_id
- title
- summary
- tasks (parseados desde formato de diccionario)

**Transformación**: Aplicar lowercase a todos los textos
**Nota**: Columna 'description' eliminada (mayoría vacía y poco útil)

In [ ]:
def extract_section(job_text, section_name):
    """
    Extrae el contenido de una sección específica del texto del job.
    Aplica lowercase.
    """
    header = section_name + "\n"
    if header not in job_text:
        return ""
    
    # Encontrar inicio de la sección
    start_idx = job_text.find(header) + len(header)
    
    # Encontrar fin de la sección (inicio de la siguiente sección o fin del texto)
    remaining_text = job_text[start_idx:]
    
    # Buscar el siguiente salto de sección (doble salto de línea seguido de MAYÚSCULAS)
    sections = ["\n\nTITLE", "\n\nSUMMARY", "\n\nDESCRIPTION", "\n\nSKILLS", 
                "\n\nTASKS", "\n\nLANGUAGES", "\n\nCERTIFICATIONS", "\n\nCOURSES"]
    
    end_idx = len(remaining_text)
    for next_section in sections:
        idx = remaining_text.find(next_section)
        if idx != -1 and idx < end_idx:
            end_idx = idx
    
    content = remaining_text[:end_idx].strip()
    
    # Aplicar lowercase
    return content.lower()

def extract_dict_list_section(job_text, section_name):
    """
    Extrae una sección que contiene una lista de diccionarios (como TASKS).
    Parsea los diccionarios y extrae solo el campo 'name' de cada uno.
    Retorna un string con los nombres separados por ' | '.
    Aplica lowercase.
    """
    header = section_name + "\n"
    if header not in job_text:
        return ""
    
    # Encontrar inicio de la sección
    start_idx = job_text.find(header) + len(header)
    remaining_text = job_text[start_idx:]
    
    # Buscar el siguiente salto de sección
    sections = ["\n\nTITLE", "\n\nSUMMARY", "\n\nDESCRIPTION", "\n\nSKILLS", 
                "\n\nTASKS", "\n\nLANGUAGES", "\n\nCERTIFICATIONS", "\n\nCOURSES"]
    
    end_idx = len(remaining_text)
    for next_section in sections:
        idx = remaining_text.find(next_section)
        if idx != -1 and idx < end_idx:
            end_idx = idx
    
    content = remaining_text[:end_idx].strip()
    
    if not content:
        return ""
    
    # Parsear los diccionarios y extraer los nombres
    names = []
    for line in content.split('\n'):
        line = line.strip()
        if line.startswith('- '):
            dict_text = line[2:]
            try:
                item_dict = ast.literal_eval(dict_text)
                name = str(item_dict.get('name', '')).strip()
                if name:
                    # Aplicar lowercase
                    names.append(name.lower())
            except:
                continue
    
    # Unir todos los nombres con separador
    return ' | '.join(names)

print("✅ Funciones extract_section y extract_dict_list_section creadas")

SyntaxError: invalid syntax (4202003744.py, line 43)

In [ ]:
# Construir tabla de descripciones
print("Construyendo tabla job_descriptions...")

descriptions_data = []

for job_id, job_text in job_listings.items():
    descriptions_data.append({
        'job_id': job_id,
        'title': extract_section(job_text, 'TITLE'),
        'summary': extract_section(job_text, 'SUMMARY'),
        'tasks': extract_dict_list_section(job_text, 'TASKS')  # Parsear TASKS correctamente
    })

job_descriptions = pd.DataFrame(descriptions_data)

print(f"\n✅ Tabla job_descriptions creada: {job_descriptions.shape}")
print(f"\nColumnas: {job_descriptions.columns.tolist()}")

# Estadísticas
print(f"\n📊 Contenido disponible:")
print(f"   - Jobs con title: {(job_descriptions['title'] != '').sum():,}")
print(f"   - Jobs con summary: {(job_descriptions['summary'] != '').sum():,}")
print(f"   - Jobs con tasks: {(job_descriptions['tasks'] != '').sum():,}")

print(f"\n📝 Nota: Columna 'description' eliminada (mayoría vacía)")
print(f"   TASKS ahora contiene solo los nombres parseados, sin formato de diccionario")

display(job_descriptions.head(10))

Construyendo tabla job_descriptions...

✅ Tabla job_descriptions creada: (21917, 5)

Columnas: ['job_id', 'title', 'summary', 'description', 'tasks']

📊 Contenido disponible:
   - Jobs con title: 21,913
   - Jobs con summary: 21,545
   - Jobs con description: 8,154
   - Jobs con tasks: 12,293


,job_id,title,summary,description,tasks
0,0,qa intégration / data analyst - salesforces s...,responsabilités :\nassurer la qualité des do...,,- {'name': 'assurer la qualité des données i...
1,1,ingénieur système,nous recherchons un ingénieur système pour n...,,- {'name': 'administrer des serveurs middlewar...
2,2,testeur qa automatisation cypress,vous avez au moins une première expérience s...,,"- {'name': 'approche axée sur les solutions',..."
3,3,ingénieur support n3 ip - paris,dans le cadre de cette mission :\nvous garanti...,,- {'name': 'apporter un support technique sur ...
4,4,business analyst moa front,nous recherchons un (e) consultant(e) ayant un...,,- {'name': 'accompagnement de la maitrise d’œu...
5,5,business analyst sap s/4,notre client est à la recherche de son procha...,,"- {'name': ""animer la communauté d'utilisateu..."
6,6,salesforce marketing cloud product owner,mission / freelance / paris / salesforce marke...,,
7,7,responsable sécurité opérationnel,opportunité freelance - responsable sécurite...,,- {'name': 'animer la mise en œuvre des politi...
8,8,architecte d'entreprise data bi,poste d'architecte d'entreprise pour rejoindre...,,"- {'name': ""conduite des\nétudes d'architectu..."
9,9,ingénieur.e qa web / mobile - (haute-savoie -...,au sein de l’équipe d’assistance à maitrise ...,,- {'name': 'amélioration des\nprocessus de re...


## 3. Tabla 2: job_skills

Extrae todas las skills (hard y soft) de cada job:
- job_id
- skill (nombre de la skill en lowercase)
- type_skill (0 = hard, 1 = soft)

**Nota**: Jobs sin skills no aparecen en esta tabla

In [ ]:
def extract_skills(job_text):
    """
    Extrae skills del texto del job.
    Retorna lista de diccionarios con {name, type}
    NO aplica lowercase en esta etapa para no romper el parsing.
    """
    # Extraer sección SKILLS sin lowercase
    header = "SKILLS\n"
    if header not in job_text:
        return []
    
    # Encontrar inicio de la sección
    start_idx = job_text.find(header) + len(header)
    remaining_text = job_text[start_idx:]
    
    # Buscar el siguiente salto de sección
    sections = ["\n\nTITLE", "\n\nSUMMARY", "\n\nDESCRIPTION", "\n\nSKILLS", 
                "\n\nTASKS", "\n\nLANGUAGES", "\n\nCERTIFICATIONS", "\n\nCOURSES"]
    
    end_idx = len(remaining_text)
    for next_section in sections:
        idx = remaining_text.find(next_section)
        if idx != -1 and idx < end_idx:
            end_idx = idx
    
    skills_content = remaining_text[:end_idx].strip()
    
    if not skills_content:
        return []
    
    skills = []
    
    # Cada skill está en una línea que empieza con "- "
    for line in skills_content.split('\n'):
        line = line.strip()
        if line.startswith('- '):
            # Remover el "- " y parsear el diccionario
            dict_text = line[2:]
            try:
                skill_dict = ast.literal_eval(dict_text)
                # Aplicar lowercase al nombre
                skill_name = str(skill_dict.get('name', '')).lower()
                skill_type = str(skill_dict.get('type', '')).lower()
                
                if skill_name and skill_type:
                    skills.append({
                        'name': skill_name,
                        'type': skill_type
                    })
            except:
                continue
    
    return skills

print("✅ Función extract_skills creada")

✅ Función extract_skills creada


In [8]:
# Construir tabla de skills
print("Construyendo tabla job_skills...")

skills_data = []

for job_id, job_text in job_listings.items():
    skills_list = extract_skills(job_text)
    
    for skill in skills_list:
        # Mapear tipo a valor numérico
        type_skill = 0 if skill['type'] == 'hard' else 1
        
        skills_data.append({
            'job_id': job_id,
            'skill': skill['name'],
            'type_skill': type_skill
        })

job_skills = pd.DataFrame(skills_data)

print(f"\n✅ Tabla job_skills creada: {job_skills.shape}")

if len(job_skills) > 0:
    print(f"\nColumnas: {job_skills.columns.tolist()}")
    
    # Estadísticas
    print(f"\n📊 Distribución de skills:")
    print(f"   - Total de skills: {len(job_skills):,}")
    print(f"   - Jobs únicos con skills: {job_skills['job_id'].nunique():,}")
    print(f"   - Skills únicas: {job_skills['skill'].nunique():,}")
    print(f"\n   Por tipo:")
    type_counts = job_skills['type_skill'].value_counts().sort_index()
    print(f"   - Hard skills (0): {type_counts.get(0, 0):,}")
    print(f"   - Soft skills (1): {type_counts.get(1, 0):,}")
    
    display(job_skills.head(15))
else:
    print("⚠️ No se encontraron skills en los jobs")

Construyendo tabla job_skills...

✅ Tabla job_skills creada: (0, 0)
⚠️ No se encontraron skills en los jobs


In [9]:
# Top 10 skills más frecuentes por tipo
print("\n🔝 Top 10 Hard Skills más frecuentes:")
hard_skills = job_skills[job_skills['type_skill'] == 0]
top_hard = hard_skills['skill'].value_counts().head(10)
for i, (skill, count) in enumerate(top_hard.items(), 1):
    print(f"   {i}. {skill}: {count:,}")

print("\n🔝 Top 10 Soft Skills más frecuentes:")
soft_skills = job_skills[job_skills['type_skill'] == 1]
top_soft = soft_skills['skill'].value_counts().head(10)
for i, (skill, count) in enumerate(top_soft.items(), 1):
    print(f"   {i}. {skill}: {count:,}")


🔝 Top 10 Hard Skills más frecuentes:


KeyError: 'type_skill'

## 4. Tabla 3: job_balancer

Crea información de balanceo para cada job:
- job_id
- num_candidates (número de candidatos que aplicaron)
- weight (peso = 1/sqrt(num_candidates))
- frequency_group (high: >100, medium: 10-100, low: <10)

In [ ]:
# Contar candidatos por job
print("Construyendo tabla job_balancer...")

# Contar ocurrencias en y_train
job_counts = y_train['job_id'].value_counts().reset_index()
job_counts.columns = ['job_id', 'num_candidates']

# Calcular peso: 1/sqrt(num_candidates)
job_counts['weight'] = 1 / np.sqrt(job_counts['num_candidates'])

# Clasificar en grupos de frecuencia
def classify_frequency(num_candidates):
    if num_candidates > 100:
        return 'high'
    elif num_candidates >= 10:
        return 'medium'
    else:
        return 'low'

job_counts['frequency_group'] = job_counts['num_candidates'].apply(classify_frequency)

job_balancer = job_counts[['job_id', 'num_candidates', 'weight', 'frequency_group']]

print(f"\n✅ Tabla job_balancer creada: {job_balancer.shape}")
print(f"\nColumnas: {job_balancer.columns.tolist()}")

# Estadísticas
print(f"\n📊 Estadísticas de balanceo:")
print(f"   - Jobs únicos: {len(job_balancer):,}")
print(f"   - Candidatos por job (promedio): {job_balancer['num_candidates'].mean():.2f}")
print(f"   - Candidatos por job (mediana): {job_balancer['num_candidates'].median():.0f}")
print(f"   - Peso promedio: {job_balancer['weight'].mean():.4f}")
print(f"\n   Distribución por frecuencia:")
freq_dist = job_balancer['frequency_group'].value_counts()
for group in ['high', 'medium', 'low']:
    count = freq_dist.get(group, 0)
    pct = (count / len(job_balancer) * 100) if len(job_balancer) > 0 else 0
    print(f"   - {group}: {count:,} ({pct:.1f}%)")

display(job_balancer.head(10))

## 5. Validación de Integridad Referencial

Verificamos que las 3 tablas estén correctamente conectadas por job_id

In [ ]:
print("="*60)
print("VALIDACIÓN DE INTEGRIDAD REFERENCIAL")
print("="*60)

# Job IDs en cada tabla
ids_descriptions = set(job_descriptions['job_id'])
ids_skills = set(job_skills['job_id'].unique())
ids_balancer = set(job_balancer['job_id'])

print(f"\n📋 Jobs por tabla:")
print(f"   - job_descriptions: {len(ids_descriptions):,}")
print(f"   - job_skills: {len(ids_skills):,}")
print(f"   - job_balancer: {len(ids_balancer):,}")

# Verificar solapamiento
print(f"\n🔗 Integridad referencial:")

# Skills debe ser subconjunto de descriptions
skills_not_in_desc = ids_skills - ids_descriptions
print(f"   - Skills sin descripción: {len(skills_not_in_desc)}")
if skills_not_in_desc:
    print(f"     ⚠️ Ejemplos: {list(skills_not_in_desc)[:5]}")

# Balancer debe ser subconjunto de descriptions
balancer_not_in_desc = ids_balancer - ids_descriptions
print(f"   - Balancer sin descripción: {len(balancer_not_in_desc)}")
if balancer_not_in_desc:
    print(f"     ⚠️ Ejemplos: {list(balancer_not_in_desc)[:5]}")

# Descriptions que no están en balancer (esperado: algunos jobs sin candidatos)
desc_not_in_balancer = ids_descriptions - ids_balancer
print(f"   - Descriptions sin balancer: {len(desc_not_in_balancer)}")
print(f"     ℹ️ Normal: jobs sin candidatos aplicantes")

# Descriptions que no están en skills (esperado: jobs sin skills)
desc_not_in_skills = ids_descriptions - ids_skills
print(f"   - Descriptions sin skills: {len(desc_not_in_skills)}")
print(f"     ℹ️ Normal: jobs sin sección SKILLS")

print(f"\n✅ Validación completada")

## 6. Visualizaciones

Gráficos para entender la distribución de datos

In [ ]:
# Visualización 1: Distribución de candidatos por job
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Distribución de num_candidates
axes[0, 0].hist(job_balancer['num_candidates'], bins=50, color='steelblue', edgecolor='black')
axes[0, 0].set_xlabel('Número de Candidatos')
axes[0, 0].set_ylabel('Número de Jobs')
axes[0, 0].set_title('Distribución de Candidatos por Job')
axes[0, 0].axvline(job_balancer['num_candidates'].median(), color='red', 
                   linestyle='--', label=f'Mediana: {job_balancer["num_candidates"].median():.0f}')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Distribución de pesos
axes[0, 1].hist(job_balancer['weight'], bins=50, color='coral', edgecolor='black')
axes[0, 1].set_xlabel('Peso (1/sqrt(num_candidates))')
axes[0, 1].set_ylabel('Número de Jobs')
axes[0, 1].set_title('Distribución de Pesos')
axes[0, 1].grid(alpha=0.3)

# 3. Grupos de frecuencia
freq_counts = job_balancer['frequency_group'].value_counts()
axes[1, 0].bar(freq_counts.index, freq_counts.values, color=['#ff6b6b', '#feca57', '#48dbfb'])
axes[1, 0].set_xlabel('Grupo de Frecuencia')
axes[1, 0].set_ylabel('Número de Jobs')
axes[1, 0].set_title('Jobs por Grupo de Frecuencia')
axes[1, 0].grid(axis='y', alpha=0.3)

# Añadir valores en las barras
for i, (group, count) in enumerate(freq_counts.items()):
    axes[1, 0].text(i, count + 50, f'{count:,}', ha='center', va='bottom', fontweight='bold')

# 4. Skills por tipo
type_counts = job_skills['type_skill'].value_counts().sort_index()
type_labels = ['Hard Skills', 'Soft Skills']
axes[1, 1].bar(type_labels, type_counts.values, color=['#4ecdc4', '#95e1d3'])
axes[1, 1].set_ylabel('Número de Skills')
axes[1, 1].set_title('Distribución por Tipo de Skill')
axes[1, 1].grid(axis='y', alpha=0.3)

# Añadir valores en las barras
for i, count in enumerate(type_counts.values):
    axes[1, 1].text(i, count + 1000, f'{count:,}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualización 2: Skills por job
skills_per_job = job_skills.groupby('job_id').size()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma
axes[0].hist(skills_per_job, bins=30, color='mediumpurple', edgecolor='black')
axes[0].set_xlabel('Número de Skills')
axes[0].set_ylabel('Número de Jobs')
axes[0].set_title('Distribución de Skills por Job')
axes[0].axvline(skills_per_job.median(), color='red', 
               linestyle='--', label=f'Mediana: {skills_per_job.median():.0f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Boxplot
axes[1].boxplot(skills_per_job, vert=True)
axes[1].set_ylabel('Número de Skills')
axes[1].set_title('Boxplot: Skills por Job')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Estadísticas de skills por job:")
print(f"   - Promedio: {skills_per_job.mean():.2f}")
print(f"   - Mediana: {skills_per_job.median():.0f}")
print(f"   - Mínimo: {skills_per_job.min()}")
print(f"   - Máximo: {skills_per_job.max()}")

## 7. Exportar Tablas

Guardamos las 3 tablas en la carpeta `Data/job_listing_feature/`

In [ ]:
# Crear carpeta de salida
output_dir = '../Data/job_listing_feature'
os.makedirs(output_dir, exist_ok=True)

print(f"📁 Carpeta de salida: {output_dir}")
print(f"\nExportando tablas...")

# Exportar las 3 tablas
job_descriptions.to_csv(f'{output_dir}/job_descriptions.csv', index=False, encoding='utf-8')
print(f"✅ job_descriptions.csv - {job_descriptions.shape[0]:,} filas")

job_skills.to_csv(f'{output_dir}/job_skills.csv', index=False, encoding='utf-8')
print(f"✅ job_skills.csv - {job_skills.shape[0]:,} filas")

job_balancer.to_csv(f'{output_dir}/job_balancer.csv', index=False, encoding='utf-8')
print(f"✅ job_balancer.csv - {job_balancer.shape[0]:,} filas")

print(f"\n🎉 Exportación completada!")

In [ ]:
# Tamaños de archivos
print("\n💾 Tamaños de archivos:")
for filename in ['job_descriptions.csv', 'job_skills.csv', 'job_balancer.csv']:
    filepath = f'{output_dir}/{filename}'
    size_mb = os.path.getsize(filepath) / 1024**2
    print(f"   - {filename}: {size_mb:.2f} MB")

## 8. Resumen Final

In [ ]:
print("="*60)
print("RESUMEN FINAL - 3 TABLAS CREADAS")
print("="*60)

print(f"\n📊 TABLA 1: job_descriptions.csv")
print(f"   - Dimensiones: {job_descriptions.shape}")
print(f"   - Columnas: {', '.join(job_descriptions.columns)}")
print(f"   - Contenido: Textos en lowercase, tasks parseados")
print(f"   - Jobs únicos: {len(job_descriptions):,}")
print(f"   - Nota: 'description' eliminada (mayoría vacía)")

print(f"\n🛠️ TABLA 2: job_skills.csv")
print(f"   - Dimensiones: {job_skills.shape}")
print(f"   - Columnas: {', '.join(job_skills.columns)}")
print(f"   - Jobs con skills: {job_skills['job_id'].nunique():,}")
print(f"   - Total skills: {len(job_skills):,}")
print(f"   - Hard skills (0): {(job_skills['type_skill'] == 0).sum():,}")
print(f"   - Soft skills (1): {(job_skills['type_skill'] == 1).sum():,}")

print(f"\n⚖️ TABLA 3: job_balancer.csv")
print(f"   - Dimensiones: {job_balancer.shape}")
print(f"   - Columnas: {', '.join(job_balancer.columns)}")
print(f"   - Jobs con candidatos: {len(job_balancer):,}")
print(f"   - Grupos de frecuencia:")
for group in ['high', 'medium', 'low']:
    count = (job_balancer['frequency_group'] == group).sum()
    print(f"     • {group}: {count:,}")

print(f"\n✅ TRANSFORMACIONES APLICADAS:")
print(f"   ✓ Lowercase en todos los textos")
print(f"   ✓ Skills hard (0) y soft (1) incluidas")
print(f"   ✓ Sin limpieza de caracteres especiales")
print(f"   ✓ Sin remoción de junior/senior")
print(f"   ✓ Pesos calculados: 1/sqrt(num_candidates)")
print(f"   ✓ TASKS parseados correctamente (sin formato de diccionario)")

print(f"\n📁 ARCHIVOS EXPORTADOS EN:")
print(f"   {os.path.abspath(output_dir)}")

print(f"\n🎯 ¡Listo para análisis y clustering!")